# Whisper large-v3 benchmark: Avaliação de Dialetos Regionais e Métricas de Precisão

## Objetivos e Metodologia
Sistemas de Reconhecimento Automático de Fala (ASR) como o OpenAI Whisper possuem alta acurácia geral, mas podem apresentar variações em sotaques regionais do português brasileiro.

Este notebook executa um pipeline de auditoria para quantificar:
- **Word Error Rate (WER):** Taxa de erro de palavras calculada comparando a transcrição do modelo com a transcrição humana de referência (ground truth) via `jiwer`.
- **Real-Time Factor (RTF):** Eficiência computacional medida pela razão entre o tempo de inferência e a duração do áudio.

---
> ⚙️ **Configuração de Hardware:** Antes de iniciar, acesse o menu *Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU*.

# 1. Configuração do ambiente

## 1.1 Instalação de Dependências e Verificação de GPU

In [ ]:
!pip install -q openai-whisper jiwer yt-dlp
import torch
print('GPU disponível:', torch.cuda.is_available())
!pip show openai-whisper | grep Version

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
GPU disponível: True
Version: 20250625


#2. Ingestão e Processamento dos Áudios
 Utilize **apenas uma** das opções abaixo para disponibilizar as amostras de áudio no ambiente antes da transcrição:
> - **2.1:** Automação de extração direta via `yt-dlp`.
> - **2.2:** Upload manual do arquivo compactado `audios.zip`.

##2.1 Coleta e Automação via yt-dlp (recomendado)


In [ ]:
#Gera a estrutura de metadados (`clips.csv`) e executa o download automatizado dos trechos de 80 segundos diretamente da fonte, padronizando o formato em 16 kHz mono.

import pandas as pd

#Definição das URLs, tempos de início e duração dos cortes dos dialetos
dados_clips = [
    {"id": "F1", "url": "https://www.youtube.com/watch?v=EXEMPLO1", "inicio": "01:15", "duracao_s": 80},
    {"id": "F2", "url": "https://www.youtube.com/watch?v=EXEMPLO2", "inicio": "03:40", "duracao_s": 80},
    {"id": "F3", "url": "https://www.youtube.com/watch?v=EXEMPLO3", "inicio": "00:50", "duracao_s": 80}
]

df = pd.DataFrame(dados_clips)
df.to_csv("clips.csv", index=False, encoding="utf-8")
print("✅ Ficheiro clips.csv gerado com sucesso!")

✅ Ficheiro clips.csv gerado com sucesso!


In [ ]:
import csv, subprocess, sys
from pathlib import Path

# Pasta de saída para os áudios cortados
OUT = Path("audios")
OUT.mkdir(exist_ok=True)

def para_seg(t):
    """Converte o tempo de formato 'MM:SS' ou 'HH:MM:SS' para segundos totais."""
    p = [int(x) for x in t.split(":")]
    s = 0
    for x in p:
        s = s * 60 + x
    return s

def hms(s):
    """Converte segundos para o formato 'H:MM:SS' exigido pelo yt-dlp."""
    return f"{s//3600}:{(s%3600)//60:02d}:{s%60:02d}"

def duracao(arq):
    """Obtém a duração real do áudio extraído usando o ffprobe."""
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                        "-of", "default=nw=1:nk=1", str(arq)], capture_output=True, text=True)
    return float(r.stdout.strip())

# Loop principal: lê o clips.csv, baixa os trechos e faz o corte de 80s
for row in csv.DictReader(open("clips.csv", encoding="utf-8")):
    fid, url = row["id"], row["url"].strip()

    if not url or not row["inicio"].strip():
        print(f"[pulado] {fid}: sem link ou início")
        continue

    alvo = OUT / f"{fid}.wav"
    if alvo.exists():
        print(f"[ok] {fid} já existe")
        continue

    ini = para_seg(row["inicio"])
    dur = int(row["duracao_s"])

    # Comando yt-dlp otimizado para baixar apenas a seção de tempo necessária e converter para 16kHz mono
    cmd = ["yt-dlp", "-x", "--audio-format", "wav",
           "--download-sections", f"*{hms(ini)}-{hms(ini+dur)}",
           "--force-keyframes-at-cuts",
           "--postprocessor-args", "ffmpeg:-ar 16000 -ac 1",
           "-o", str(OUT / f"{fid}.%(ext)s"), url]

    print(f"[baixando] {fid} {row['inicio']} +{dur}s")
    if subprocess.run(cmd).returncode != 0:
        print(f"[ERRO] {fid}: falhou (veja a mensagem acima)")
        continue

    try:
        d = duracao(alvo)
        aviso = "" if abs(d - dur) <= 1 else "  <-- CONFERIR: duração diferente de 80 s"
        print(f"[feito] {fid}: {d:.1f} s{aviso}")
    except Exception as e:
        print(f"[feito] {fid} (não consegui conferir a duração: {e})")

## 2.2 Upload de Arquivo Zip

In [ ]:
#Realiza a extração local do ficheiro compactado contendo as amostras de áudio previamente cortadas.
from google.colab import files
import zipfile

enviado = files.upload()  # selecione audios.zip
zipfile.ZipFile('audios.zip').extractall('audios')
!ls audios

## 2.3 Transcrição via Whisper e cálculo RTF

In [ ]:
"""
Etapa 1: Transcreve todos os .wav da pasta 'audios' com o Whisper
e mede o tempo de processamento (com parâmetros anti-alucinação).
"""
import csv
import statistics
import time
from pathlib import Path
import wave

import torch
import whisper

PASTA_AUDIOS = Path("audios")
PASTA_SAIDA = Path("saida")
MODELO = "large-v3"
EXECUCOES = 3

PASTA_SAIDA.mkdir(exist_ok=True)

# Procura ficheiros .wav inclusive em subpastas
arquivos = sorted(PASTA_AUDIOS.rglob("*.wav"), key=lambda p: int(p.stem[1:]))

if not arquivos:
    raise SystemExit(f"Nenhum arquivo .wav encontrado em '{PASTA_AUDIOS}'.")

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {dispositivo} | Modelo: {MODELO} | Whisper: {getattr(whisper, '__version__', '?')}")

# Carrega o modelo
model = whisper.load_model(MODELO, device=dispositivo)

# 1. Warm-up (Aquecimento da GPU)
print("Executando aquecimento da GPU...")
if arquivos:
    if dispositivo == "cuda":
        torch.cuda.synchronize()
    model.transcribe(str(arquivos[0]), language="pt", condition_on_previous_text=False)
    if dispositivo == "cuda":
        torch.cuda.synchronize()

tempos_relatorio = []

# 2. Loop de Transcrição
for arq in arquivos:
    frag = arq.stem

    # Obter duração do áudio em segundos
    with wave.open(str(arq), "rb") as wf:
        duracao_s = wf.getnframes() / float(wf.getframerate())

    execucoes_tempos = []
    texto_transcrito = ""

    for i in range(EXECUCOES):
        if dispositivo == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()

        # Transcrição com parâmetros que IMPEDEM ALUCINAÇÃO E LOOPS
        resultado = model.transcribe(
            str(arq),
            language="pt",
            condition_on_previous_text=False,  # Bloqueia a repetição de frases em loop
            temperature=0.0,                   # Torna o resultado determinístico
            no_speech_threshold=0.6            # Evita inventar texto no silêncio
        )

        if dispositivo == "cuda":
            torch.cuda.synchronize()

        t1 = time.perf_counter()

        execucoes_tempos.append(t1 - t0)
        texto_transcrito = resultado["text"].strip()

    mediana_s = statistics.median(execucoes_tempos)
    rtf = mediana_s / duracao_s if duracao_s > 0 else 0

    # Grava o texto gerado pelo Whisper na pasta 'saida'
    (PASTA_SAIDA / f"{frag}.txt").write_text(texto_transcrito, encoding="utf-8")

    tempos_relatorio.append({
        "fragmento": frag,
        "duracao_s": round(duracao_s, 2),
        "mediana_s": round(mediana_s, 2),
        "rtf": round(rtf, 4)
    })

    print(f"{frag}: mediana {mediana_s:.1f}s | RTF {rtf:.2f}")

# Salva a tabela de tempos
with open(PASTA_SAIDA / "tempos.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["fragmento", "duracao_s", "mediana_s", "rtf"])
    writer.writeheader()
    writer.writerows(tempos_relatorio)

print("\nPronto! Etapa 1 concluída. Veja os arquivos gerados em 'saida/'.")

Dispositivo: cuda | Modelo: large-v3 | Whisper: 20250625


100%|██████████████████████████████████████| 2.88G/2.88G [00:25<00:00, 121MiB/s]


Executando aquecimento da GPU...
F1: mediana 12.8s | RTF 0.16
F3: mediana 17.0s | RTF 0.21
F4: mediana 14.3s | RTF 0.18
F5: mediana 14.5s | RTF 0.18
F6: mediana 12.8s | RTF 0.16
F7: mediana 18.0s | RTF 0.23
F8: mediana 16.1s | RTF 0.20
F9: mediana 16.2s | RTF 0.20
F10: mediana 14.0s | RTF 0.17
F11: mediana 20.3s | RTF 0.25
F12: mediana 20.2s | RTF 0.25
F13: mediana 19.3s | RTF 0.24
F14: mediana 17.7s | RTF 0.22

Pronto! Etapa 1 concluída. Veja os arquivos gerados em 'saida/'.


# 3. Ingestão de Ground Truth (Transcrições Humanas de Referência)

##3.1 Importação e Descompactação do Arquivo de Referências (`GSGT_transcricoes.zip`)

In [ ]:
from google.colab import files
import zipfile

enviado = files.upload()  # selecione GSGT_transcricoes.zip
zipfile.ZipFile('GSGT_transcricoes.zip').extractall('referencias')
!ls referencias

Saving GSGT_transcricoes.zip to GSGT_transcricoes.zip
F10.txt  F12.txt  F14.txt  F3.txt  F5.txt  F7.txt  F9.txt
F11.txt  F13.txt  F1.txt   F4.txt  F6.txt  F8.txt


# 4. Cálculo do Word Error Rate (WER) e Mapeamento de Erros

Calculo da acurácia do modelo comparando a transcrição gerada pelo Whisper com a **Ground Truth humana** através do alinhamento de distância de Levenshtein (via biblioteca `jiwer`).


**Metodologia de Limpeza**<br>                                                 Para garantir que a medição seja estritamente focada em acertos e erros de palavras — e não em diferenças de formatação —, a função de normalização é aplicada diretamente em memória durante a leitura dos arquivos:
- Conversão para letras minúsculas (`lower()`).
- Remoção de pontuações e símbolos especiais.
- Eliminação de quebras de linha e múltiplos espaços.

O processamento gera o arquivo `saida/wer_detalhado.csv` contendo as métricas de **Substituições (S)**, **Deleções/Remoções (D)**, **Inserções (I)** e acertos (**Hits**) para cada amostra regional.

In [ ]:
"""
Etapa 2: Processamento e cálculo das métricas de erro de transcrição (WER).
Entradas:
  - referencias/{frag}.txt -> Transcrição humana de referência
  - saida/{frag}.txt       -> Transcrição de hipótese (Whisper)
Saída:
  - saida/wer_detalhado.csv -> Mapeamento de erros por fragmento
"""
import csv
import re
from pathlib import Path
import jiwer

PASTA_REF = Path("referencias")
PASTA_SAIDA = Path("saida")


def normalizar(texto: str) -> str:
    """Aplica minúsculas, remove pontuações e normaliza espaços mantendo acentos."""
    texto = texto.lower()
    texto = re.sub(r"[^\w\s]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()


def calcular_wer_fragmento(path_ref: Path, path_hip: Path) -> dict:
    """Calcula métricas detalhadas de WER para um único par referência/hipótese."""
    ref = normalizar(path_ref.read_text(encoding="utf-8"))
    hip = normalizar(path_hip.read_text(encoding="utf-8"))

    r = jiwer.process_words(ref, hip)

    return {
        "fragmento": path_ref.stem,
        "palavras_referencia": len(ref.split()),
        "palavras_hipotese": len(hip.split()),
        "wer": round(r.wer, 4),
        "substituicoes": r.substitutions,
        "remocoes": r.deletions,
        "insercoes": r.insertions,
        "hits": r.hits,
    }


def executar_calculo_wer() -> list[dict]:
    PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
    resultados = []

    for ref_path in sorted(PASTA_REF.glob("*.txt"), key=lambda p: int(p.stem[1:])):
        frag = ref_path.stem
        hip_path = PASTA_SAIDA / f"{frag}.txt"

        if not hip_path.exists():
            print(f"⚠️ {frag}: sem transcrição em '{hip_path}', pulando.")
            continue

        m = calcular_wer_fragmento(ref_path, hip_path)
        resultados.append(m)
        print(f"✓ {frag}: WER {m['wer']:.1%} (S:{m['substituicoes']}, D:{m['remocoes']}, I:{m['insercoes']})")

    if not resultados:
        raise SystemExit("Erro: Nenhum par de transcrição foi encontrado.")

    # Exporta resultados do WER
    out_csv = PASTA_SAIDA / "wer_detalhado.csv"
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(resultados[0].keys()))
        writer.writeheader()
        writer.writerows(resultados)

    print(f"\n✅ Bloco WER concluído: '{out_csv}' gerado.")
    return resultados


if __name__ == "__main__":
    executar_calculo_wer()

✓ F1: WER 5.1% (S:5, D:0, I:3)
✓ F3: WER 10.4% (S:12, D:8, I:4)
✓ F4: WER 10.1% (S:10, D:9, I:3)
✓ F5: WER 16.4% (S:12, D:18, I:5)
✓ F6: WER 18.8% (S:19, D:14, I:2)
✓ F7: WER 8.7% (S:15, D:2, I:2)
✓ F8: WER 19.2% (S:25, D:11, I:5)
✓ F9: WER 16.5% (S:20, D:12, I:4)
✓ F10: WER 21.5% (S:15, D:11, I:12)
✓ F11: WER 15.6% (S:14, D:15, I:7)
✓ F12: WER 29.0% (S:40, D:4, I:23)
✓ F13: WER 20.6% (S:43, D:7, I:2)
✓ F14: WER 25.2% (S:28, D:8, I:16)

✅ Bloco WER concluído: 'saida/wer_detalhado.csv' gerado.


#5. Análise de métricas e análise qualitativa

In [ ]:
!pip install -q google-genai

In [ ]:
import json
import time
from pathlib import Path
from google import genai
from google.colab import userdata

# 1. Configuração do Cliente Gemini
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

# 2. Definição de Pastas
PASTA_SAIDA = Path("saida")
PASTA_SAIDA.mkdir(exist_ok=True)

# 3. Descompacta o ZIP
!unzip -o /content/GSGT_transcricoes.zip -d /content/

# Procura todos os ficheiros .txt extraídos e ordena pelo número do arquivo (F1 a F14)
ficheiros_txt = sorted(
    list(Path("/content").glob("F*.txt")),
    key=lambda x: int("".join(filter(str.isdigit, x.stem)))
    if "".join(filter(str.isdigit, x.stem))
    else 0,
)

print(
    f"📖 Encontrados {len(ficheiros_txt)} ficheiros para reprocessar com o PROMPT CORRETO."
)

# 4. PROMPT OFICIAL DE LINGUÍSTICA COMPUTICIONAL / BENCHMARK WHISPER
SYSTEM_INSTRUCTION = """Você é um Engenheiro de IA Sênior e Especialista em Linguística Computacional.
Sua tarefa é analisar os dados de um benchmark do modelo OpenAI Whisper em dialectos e sotaques do Português Brasileiro.

Diretrizes de Análise:
1. O Grupo Controle (geralmente F1) deve ser usado como referência de linha de base.
2. Avalie como as métricas de Substituição (S), Deleção (D) e Inserção (I) se correlacionam com traços de fala regionais (ritmo, supressão de r/s final, chiamento, vogais).
3. Seja conciso, técnico e direto ao ponto. Suas respostas serão consumidas por um dashboard executivo."""


# 5. Função de chamada com retry reforçado e tempo de espera maior para não estourar a API
def analisar_benchmark(texto_fragmento, max_tentativas=5):
    prompt_completo = f"{SYSTEM_INSTRUCTION}\n\nAnalise os seguintes dados do benchmark:\n{texto_fragmento}"
    for tentativa in range(max_tentativas):
        try:
            resposta = client.models.generate_content(
                model="gemini-3-flash-preview", contents=[prompt_completo]
            )
            return resposta.text
        except Exception as e:
            erro_str = str(e)
            if "429" in erro_str or "503" in erro_str:
                print(
                    f"⚠️ Cota atingida. Aguardando 15 segundos (tentativa {tentativa+1}/{max_tentativas})..."
                )
                time.sleep(15)
            else:
                print(f"❌ Erro inesperado: {e}")
                return None
    return None


# 6. Reprocessa todos os fragmentos com a análise correta
base_consolidada = []

print("\n🚀 Executando o reprocessamento completo do benchmark...\n")

for caminho_txt in ficheiros_txt:
    try:
        texto = caminho_txt.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        texto = caminho_txt.read_text(encoding="cp1252")

    resultado = analisar_benchmark(texto)

    if resultado:
        print(f"✓ {caminho_txt.name} analisado com sucesso (Linguística)!")
        base_consolidada.append(
            {"ficheiro": caminho_txt.name, "analise": resultado}
        )
    else:
        print(f"⚠️ Falha no processamento de {caminho_txt.name}")

    # Pausa de 5 segundos entre requisições para evitar rate limit
    time.sleep(5)

# 7. Salva a Base Dashboard Final Corrigida
(PASTA_SAIDA / "base_dashboard.json").write_text(
    json.dumps(base_consolidada, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(
    f"\n✅ REPROCESSAMENTO CONCLUÍDO! O arquivo 'saida/base_dashboard.json' foi reconstruído com as métricas certas do Whisper."
)

Archive:  /content/GSGT_transcricoes.zip
  inflating: /content/F1.txt         
  inflating: /content/F10.txt        
  inflating: /content/F11.txt        
  inflating: /content/F12.txt        
  inflating: /content/F13.txt        
  inflating: /content/F14.txt        
  inflating: /content/F3.txt         
  inflating: /content/F4.txt         
  inflating: /content/F5.txt         
  inflating: /content/F6.txt         
  inflating: /content/F7.txt         
  inflating: /content/F8.txt         
  inflating: /content/F9.txt         
📖 Encontrados 13 ficheiros para reprocessar com o PROMPT CORRETO.

🚀 Executando o reprocessamento completo do benchmark...

✓ F1.txt analisado com sucesso (Linguística)!
✓ F3.txt analisado com sucesso (Linguística)!
✓ F4.txt analisado com sucesso (Linguística)!
✓ F5.txt analisado com sucesso (Linguística)!
✓ F6.txt analisado com sucesso (Linguística)!
✓ F7.txt analisado com sucesso (Linguística)!
✓ F8.txt analisado com sucesso (Linguística)!
✓ F9.txt analisado 

###validação + conversão json em csv/ pandas

In [ ]:
import json
from pathlib import Path

caminho_json = Path("saida/base_dashboard.json")

if caminho_json.exists():
    base = json.loads(caminho_json.read_text(encoding="utf-8"))
    print(f"✅ Total de itens salvos: {len(base)} / 13\n")

    for idx, item in enumerate(base, start=1):
        nome = item.get("ficheiro") or item.get("fragmento")
        tamanho = len(item.get("analise", ""))
        status = (
            "OK"
            if tamanho > 50
            else "⚠️ ATENÇÃO (resposta muito curta ou vazia)"
        )
        print(f"[{idx}] {nome} -> {tamanho} caracteres | Status: {status}")
else:
    print("❌ Arquivo 'saida/base_dashboard.json' não foi encontrado!")

In [ ]:
import json
import pandas as pd
from pathlib import Path

# Carrega o JSON consolidado
caminho_json = Path("saida/base_dashboard.json")
base = json.loads(caminho_json.read_text(encoding="utf-8"))

# Converte em DataFrame e exporta para CSV
df = pd.DataFrame(base)
df.to_csv("saida/base_dashboard.csv", index=False, encoding="utf-8-sig")

print("📊 Dataset em CSV gerado com sucesso em 'saida/base_dashboard.csv'!")

📊 Dataset em CSV gerado com sucesso em 'saida/base_dashboard.csv'!


# 6. Consolidação do Dataset para Análise / Dashboard
Cruzamento dos metadados sociolinguísticos dos falantes (região, dialeto e cidade) com as métricas de **WER** e **RTF** obtidas.

O resultado é exportado no arquivo `base_dashboard.csv`, pronto para alimentação de relatórios e visualizações analíticas.

In [ ]:
# Versão alternativa da análise, com as métricas reais do jiwer no prompt.
# Não foi usada nos resultados do relatório: rodar esta célula refaz as análises e
# sobrescreve saida/base_dashboard.json. Mude para True apenas se quiser isso.
RODAR_ANALISE_COM_METRICAS = False

if RODAR_ANALISE_COM_METRICAS:
    import os
    import json
    import pandas as pd
    from google import genai
    from google.genai import types
    from google.colab import userdata
    from pydantic import BaseModel, Field

    # 1. Inicializar cliente (usa GEMINI_API_KEY dos Colab Secrets)
    client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

    # 2. Schema estruturado: impede cabeçalhos inventados, datas falsas e placeholders
    class AnaliseFragmento(BaseModel):
        dialeto_identificado: str = Field(description="Nome do dialeto ou variante regional analisada")
        diagnostico_substituicoes: str = Field(description="Análise linguística das substituições reais observadas entre referência e hipótese")
        diagnostico_delecoes: str = Field(description="Análise linguística das remoções/deleções reais observadas")
        diagnostico_insercoes: str = Field(description="Análise linguística das inserções reais observadas")
        sintese_executiva: str = Field(description="Conclusão objetiva grounded nos números reais do WER, sem estimativas")
        analise_markdown: str = Field(description="Relatório completo em Markdown bem formatado para exibição no dashboard, usando exclusivamente os números fornecidos")

    # 3. Carregar métricas calibradas
    df_wer = pd.read_csv('saida/wer_detalhado.csv')
    resultados_dashboard = []

    system_instruction = """
    Você é um especialista em Linguística Computacional e avaliação de modelos ASR (Whisper) no Português Brasileiro (metodologia ALiB).
    REGRAS ESTRITAS:
    1. Você receberá as métricas EXATAS já calculadas pelo jiwer (WER, Substituições, Deleções, Inserções e Hits), além da transcrição humana de referência e da saída do Whisper.
    2. É PROIBIDO estimar, prever ou supor percentuais de erro (nunca use 'espera-se', 'métricas estimadas' ou faixas hipotéticas).
    3. É PROIBIDO inventar metadados como datas, nomes de analistas ou cabeçalhos fictícios.
    4. Baseie cada ponto da sua explicação nas divergências reais entre o texto de referência e a hipótese do Whisper fornecidos.
    """

    for _, row in df_wer.iterrows():
        frag = row['fragmento']
        ref_path = f"referencias/{frag}.txt"
        hyp_path = f"saida/{frag}.txt"

        if not (os.path.exists(ref_path) and os.path.exists(hyp_path)):
            print(f"Pulando {frag}: arquivo ausente.")
            continue

        with open(ref_path, 'r', encoding='utf-8') as f:
            ref_texto = f.read().strip()
        with open(hyp_path, 'r', encoding='utf-8') as f:
            hyp_texto = f.read().strip()

        wer_pct = round(row['wer'] * 100, 2)

        prompt = f"""
        Analise o fragmento {frag} com base nos dados reais abaixo:

        MÉTRICAS CALIBRADAS (jiwer):
        - WER Real: {wer_pct}% ({row['wer']})
        - Palavras na Referência: {int(row['palavras_referencia'])}
        - Palavras na Hipótese (Whisper): {int(row['palavras_hipotese'])}
        - Acertos (Hits): {int(row['hits'])}
        - Substituições (S): {int(row['substituicoes'])}
        - Deleções/Remoções (D): {int(row['remocoes'])}
        - Inserções (I): {int(row['insercoes'])}

        TRANSCRIÇÃO HUMANA (REFERÊNCIA):
        {ref_texto}

        TRANSCRIÇÃO DO MODELO (WHISPER):
        {hyp_texto}
        """

        response = client.models.generate_content(
            model='gemini-3-flash-preview',
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                response_mime_type="application/json",
                response_schema=AnaliseFragmento,
                temperature=0.2,
            ),
        )

        analise_estruturada = json.loads(response.text)

        # Unifica métricas numéricas + análise grounded no mesmo registro
        registro = {
            "ficheiro": f"{frag}.txt",
            "fragmento": frag,
            "wer": float(row['wer']),
            "wer_percentual": wer_pct,
            "palavras_referencia": int(row['palavras_referencia']),
            "palavras_hipotese": int(row['palavras_hipotese']),
            "hits": int(row['hits']),
            "substituicoes": int(row['substituicoes']),
            "remocoes": int(row['remocoes']),
            "insercoes": int(row['insercoes']),
            "dialeto": analise_estruturada["dialeto_identificado"],
            "diagnostico_substituicoes": analise_estruturada["diagnostico_substituicoes"],
            "diagnostico_delecoes": analise_estruturada["diagnostico_delecoes"],
            "diagnostico_insercoes": analise_estruturada["diagnostico_insercoes"],
            "sintese_executiva": analise_estruturada["sintese_executiva"],
            "analise": analise_estruturada["analise_markdown"]
        }

        resultados_dashboard.append(registro)
        print(f"Concluído: {frag} (WER: {wer_pct}%)")

    # 4. Salvar JSON final unificado
    with open('saida/base_dashboard.json', 'w', encoding='utf-8') as f:
        json.dump(resultados_dashboard, f, ensure_ascii=False, indent=2)

    print("\nPronto! Novo base_dashboard.json gerado com sucesso.")
else:
    print('Célula ignorada (RODAR_ANALISE_COM_METRICAS = False).')

In [ ]:
import json
import re
import pandas as pd

# 1. Carrega o JSON original com as 13 análises e o CSV calibrado
caminho_json = 'saida/base_dashboard.json'
caminho_csv = 'saida/wer_detalhado.csv'

with open(caminho_json, 'r', encoding='utf-8') as f:
    dados_json = json.load(f)

df_wer = pd.read_csv(caminho_csv)
wer_por_id = df_wer.set_index('fragmento').to_dict(orient='index')

# 2. Cruza os dois arquivos e higieniza os cabeçalhos/estimativas
for item in dados_json:
    codigo = item['ficheiro'].replace('.txt', '').strip()

    if codigo in wer_por_id:
        m = wer_por_id[codigo]
        wer_pct = round(m['wer'] * 100, 2)
        s, d, i, hits = int(m['substituicoes']), int(m['remocoes']), int(m['insercoes']), int(m['hits'])

        # Injeta as métricas calibradas no objeto
        item['fragmento'] = codigo
        item['wer'] = float(m['wer'])
        item['wer_percentual'] = wer_pct
        item['palavras_referencia'] = int(m['palavras_referencia'])
        item['palavras_hipotese'] = int(m['palavras_hipotese'])
        item['hits'] = hits
        item['substituicoes'] = s
        item['remocoes'] = d
        item['insercoes'] = i

        # Limpa datas fictícias e rótulos de "estimativa" do texto
        texto = item['analise']
        texto = re.sub(r'\*\*Data:\*\*.*?\n', '', texto)
        texto = re.sub(r'\*\*Analista:\*\*.*?\n', '', texto)
        texto = texto.replace('(Métricas Estimadas)', f'(WER Real: {wer_pct}% | S:{s}, D:{d}, I:{i})')
        texto = texto.replace('(Estimativa vs. Baseline F1)', f'(WER Real: {wer_pct}% vs. Baseline F1)')

        banner = f"> **Métricas Calibradas (jiwer):** WER **{wer_pct}%** | Acertos: **{hits}** | Substituições: **{s}** | Deleções: **{d}** | Inserções: **{i}**\n\n"
        item['analise'] = banner + texto.strip()
    else:
        print(f"Aviso: {codigo} não encontrado no CSV.")

# 3. Salva o arquivo final pronto para o dashboard
with open('saida/base_dashboard_final.json', 'w', encoding='utf-8') as f:
    json.dump(dados_json, f, ensure_ascii=False, indent=2)

print(f"Pronto! {len(dados_json)} fragmentos consolidados em 'saida/base_dashboard_final.json'.")

Pronto! 13 fragmentos consolidados em 'saida/base_dashboard_final.json'.
